In [ ]:
from datetime import datetime
import pandas as pd
from meteostat import hourly

# 1. Das Einstellungs-Modul von Meteostat importieren
from meteostat import config

def fetch_hourly_weather_data():
    # 2. Die Notbremse für große Zeiträume VOR der Abfrage deaktivieren
    config.block_large_requests = False

    # Hier fängt dein Zeitraum an (z.B. ab 1980)
    start = datetime(1980, 1, 1)
    end = datetime(2023, 12, 31, 23, 59) 

    stations = {
        "München": '10865',
        "Essen": '10410',
        "Schleswig": '10035',
        "Nürnberg": '10763'
    }

    all_data = []

    print("Starte stündlichen Wetter-Download über Station-IDs...")

    for city, station_id in stations.items():
        print(f"Lade Daten für {city} (Station {station_id})...")
        
        # HIER GEÄNDERT: jetzt hourly statt Daily
        data = hourly(station_id, start, end).fetch()
        
        if data is not None and not data.empty:
            data = data.reset_index()
            data['Stadt'] = city
            all_data.append(data)
        else:
            print(f"-> Achtung: Keine Daten für {city} gefunden. Überspringe...")

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        
        print("\n-> Info: Diese Spalten hat die API tatsächlich geliefert:")
        print(final_df.columns.tolist())
        
        # 2. HIER GEÄNDERT: Neue Spalten für stündliche Daten
       
        desired_columns = ['time', 'Stadt', 'temp', 'rhum', 'prcp', 'wspd', 'wdir', 'wpgt', 'tsun', 'pres', 'cldc']
        existing_columns = [col for col in desired_columns if col in final_df.columns]
        
        final_df = final_df[existing_columns]
        
        # 3. HIER GEÄNDERT: Passendes Umbenennen
        rename_dict = {
            'time': 'Datum_Uhrzeit',
            'temp': 'Temperatur_C',
            'rhum': 'Luftfeuchtigkeit_Prozent',
            'prcp': 'Niederschlag_mm',
            'wspd': 'Wind_kmh',
            'wdir': 'Windrichtung_Grad',  # NEU: Windrichtung in Grad (0-360)
            'wpgt': 'Windboeen_kmh',      # NEU: Windböen-Spitze in km/h
            'tsun': 'Sonnenschein_min',
            'pres': 'Luftdruck_hPa',
            'cldc': 'Bewoelkung_okta'
        }
        final_df.rename(columns=rename_dict, inplace=True, errors='ignore') 
        
        print("\nDownload abgeschlossen! Datenaufbereitung erfolgreich.")
        return final_df
    else:
        print("\nFehler: Es konnten gar keine Daten geladen werden.")
        return None

# ==========================================
# TESTLAUF
# ==========================================
if __name__ == "__main__":
    hourly_df = fetch_hourly_weather_data()
    
    if hourly_df is not None:
        print(f"\nGesamtzahl der geladenen Datensätze (Zeilen): {len(hourly_df)}")
        print("\nHier sind die ersten 5 Zeilen unseres stündlichen Datensatzes:")
        display(hourly_df.head())
        
        #hourly_df.to_csv("wetter_hourly_2023.csv", index=False)

Starte stündlichen Wetter-Download über Station-IDs...
Lade Daten für München (Station 10865)...
Lade Daten für Essen (Station 10410)...
Lade Daten für Schleswig (Station 10035)...
Lade Daten für Nürnberg (Station 10763)...

-> Info: Diese Spalten hat die API tatsächlich geliefert:
['time', 'temp', 'rhum', 'prcp', 'snwd', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'cldc', 'coco', 'Stadt']

Download abgeschlossen! Datenaufbereitung erfolgreich.

Gesamtzahl der geladenen Datensätze (Zeilen): 1508589

Hier sind die ersten 5 Zeilen unseres stündlichen Datensatzes:


,Datum_Uhrzeit,Stadt,Temperatur_C,Luftfeuchtigkeit_Prozent,Niederschlag_mm,Wind_kmh,Windrichtung_Grad,Windboeen_kmh,Sonnenschein_min,Luftdruck_hPa,Bewoelkung_okta
0,1980-01-01 06:00:00,München,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1010.0,8
1,1980-01-01 18:00:00,München,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1010.2,5
2,1980-01-02 06:00:00,München,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1011.9,7
3,1980-01-02 18:00:00,München,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1017.5,8
4,1980-01-03 06:00:00,München,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1023.0,5


In [19]:
hourly_df.to_csv("wetter_3cities_2023.csv", index=False)